<a href="https://colab.research.google.com/github/gabriel-tfg/TFM_RAG_Pipeline/blob/main/PruebasTFM6_Meta_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "huggingface_hub>=0.34.0" "transformers>=4.55.0" "sentence-transformers>=3.0.0" faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.2 MB/s eta 0:00:00


1) Query configuration

In [ ]:
import requests
import xml.etree.ElementTree as ET
import json
import re
from collections import defaultdict

QUERY_1 = """
"meta-analysis"[Publication Type] AND ("Menopause"[MeSH]) AND
("Quality of Life"[MeSH]
OR "Sleep Wake Disorders"[MeSH]
OR "Stress, Psychological"[MeSH]
OR "Fatigue"[MeSH]
OR "Anxiety"[MeSH]
OR "Cognition Disorders"[MeSH]
OR "Cognitive Dysfunction"[MeSH]
OR "Executive Function"[MeSH]
OR "Memory"[MeSH]
OR cognit*[tiab]
OR fatigue[tiab])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_2 = """
("meta-analysis"[Publication Type] AND "menopause/psychology"[MeSH Terms])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_3 = """
("meta-analysis"[Publication Type] AND "menopause/physiology"[MeSH Terms])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

ACTIVE_QUERY = QUERY_1   # cambiar a QUERY_2 o QUERY_3 para pruebas
RETMAX = 10

2) Pubmed search helpers

In [ ]:
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH_URL  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
ELINK_URL   = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"

def pubmed_search(query, retmax=20):
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json"
    }
    data = requests.get(ESEARCH_URL, params=params).json()
    return data["esearchresult"]["idlist"]

def pubmed_fetch_xml(pmids):
    if not pmids:
        return None
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml"
    }
    xml_text = requests.get(EFETCH_URL, params=params).content
    return ET.fromstring(xml_text)

def parse_pubmed_articles(root):
    docs = []
    if root is None:
        return docs

    for i, article in enumerate(root.findall(".//PubmedArticle")):
        pmid_elem = article.find(".//PMID")
        title_elem = article.find(".//ArticleTitle")
        abstract_elems = article.findall(".//Abstract/AbstractText")

        pmid = pmid_elem.text.strip() if pmid_elem is not None and pmid_elem.text else None
        if title_elem is None:
            continue

        title = "".join(title_elem.itertext()).strip()
        abstract = " ".join("".join(a.itertext()).strip() for a in abstract_elems) if abstract_elems else ""

        text = f"{title}\n\n{abstract}".strip()

        docs.append({
            "id": f"meta_{pmid}" if pmid else f"meta_{i}",
            "pmid": pmid,
            "type": "meta-analysis",
            "title": title,
            "text": text
        })

    return docs

# Buscar metaanálisis
meta_pmids = pubmed_search(ACTIVE_QUERY, retmax=RETMAX)
print("Meta-analysis PMIDs:", len(meta_pmids))

meta_root = pubmed_fetch_xml(meta_pmids)
meta_docs = parse_pubmed_articles(meta_root)

print("Meta-analysis docs:", len(meta_docs))
print(meta_docs[0]["title"] if meta_docs else "No docs")

Meta-analysis PMIDs: 10
Meta-analysis docs: 10
Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis.


3) Link PubMed -> PMC

In [ ]:
import requests
import time

ID_CONVERTER_URL = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"

def get_pmcid_for_pmid(pmid, tool="rag_pipeline", email="tu_email@ejemplo.com"):
    params = {
        "ids": str(pmid),
        "format": "json",
        "tool": tool,
        "email": email
    }

    try:
        r = requests.get(ID_CONVERTER_URL, params=params, timeout=20)
        r.raise_for_status()

        # Ver qué tipo de contenido devuelve realmente
        content_type = r.headers.get("Content-Type", "")

        if "json" not in content_type.lower():
            print(f"[Aviso] PMID {pmid}: la respuesta no es JSON.")
            print("Content-Type:", content_type)
            print("Respuesta:", r.text[:300])
            return None

        if not r.text.strip():
            print(f"[Aviso] PMID {pmid}: respuesta vacía.")
            return None

        data = r.json()
        records = data.get("records", [])
        if not records:
            print(f"[Aviso] PMID {pmid}: sin records.")
            return None

        pmcid = records[0].get("pmcid")
        return pmcid

    except requests.exceptions.RequestException as e:
        print(f"[Error HTTP] PMID {pmid}: {e}")
        return None
    except ValueError as e:
        print(f"[Error JSON] PMID {pmid}: {e}")
        print("Respuesta recibida:", r.text[:300])
        return None

meta_to_pmc = {}
for d in meta_docs:
    if d.get("pmid"):
        meta_to_pmc[d["pmid"]] = get_pmcid_for_pmid(d["pmid"])
        time.sleep(0.3)

print(meta_to_pmc)

{'41531400': 'PMC12835450', '41448220': None, '41401249': None, '41341454': 'PMC12669000', '41161257': 'PMC12597306', '41129871': None, '41066270': None, '40971657': None, '40742785': None, '40718787': 'PMC12296567'}


4) PMC XML retrieval + reference extraction

In [ ]:
from collections import defaultdict
import requests
import xml.etree.ElementTree as ET
import time
import re

PMC_OAI_URL = "https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/"

def fetch_pmc_xml_oai(pmcid):
    if not pmcid:
        return None

    pmc_num = pmcid.replace("PMC", "").strip()

    params = {
        "verb": "GetRecord",
        "identifier": f"oai:pubmedcentral.nih.gov:{pmc_num}",
        "metadataPrefix": "pmc"
    }

    try:
        r = requests.get(PMC_OAI_URL, params=params, timeout=30)
        r.raise_for_status()

        if not r.text.strip():
            print(f"[Aviso] {pmcid}: respuesta vacía")
            return None

        return r.text

    except requests.exceptions.RequestException as e:
        print(f"[Error HTTP] {pmcid}: {e}")
        return None

def extract_article_xml_from_oai(oai_xml_text):
    if not oai_xml_text:
        return None

    try:
        root = ET.fromstring(oai_xml_text)
    except Exception as e:
        print(f"[Error XML OAI] {e}")
        return None

    ns = {
        "oai": "http://www.openarchives.org/OAI/2.0/"
    }

    metadata = root.find(".//oai:metadata", ns)
    if metadata is None or len(metadata) == 0:
        print("[Aviso] No se encontró bloque <metadata> en la respuesta OAI")
        return None

    article_elem = list(metadata)[0]

    try:
        return ET.tostring(article_elem, encoding="unicode")
    except Exception as e:
        print(f"[Error serializando article XML] {e}")
        return None

def extract_references_from_pmc_xml(article_xml_text):
    refs = []
    if not article_xml_text:
        return refs

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando article XML] {e}")
        return refs

    for i, ref in enumerate(root.findall(".//{*}ref")):
        ref_text = " ".join(t.strip() for t in ref.itertext() if t and t.strip())

        pmid = None
        doi = None
        year = None
        first_author = None

        ref_id = ref.attrib.get("id", f"ref_{i}")

        for pubid in ref.findall(".//{*}pub-id"):
            id_type = pubid.attrib.get("pub-id-type", "").lower()
            if pubid.text:
                if id_type == "pmid" and pmid is None:
                    pmid = pubid.text.strip()
                elif id_type == "doi" and doi is None:
                    doi = pubid.text.strip()

        year_match = re.search(r"\b(19|20)\d{2}\b", ref_text)
        if year_match:
            year = year_match.group(0)

        surname_elem = ref.find(".//{*}surname")
        if surname_elem is not None and surname_elem.text:
            first_author = surname_elem.text.strip().lower()

        refs.append({
            "ref_id": ref_id,
            "pmid": pmid,
            "doi": doi,
            "year": year,
            "first_author": first_author,
            "ref_text": ref_text
        })

    return refs

4.1) Included study detection (heuristic)

In [ ]:
import xml.etree.ElementTree as ET

def extract_candidate_sections(article_xml_text):
    sections = []
    if not article_xml_text:
        return sections

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando XML para secciones] {e}")
        return sections

    for sec in root.findall(".//{*}sec"):
        title_elem = sec.find("./{*}title")
        title = ""
        if title_elem is not None:
            title = " ".join(t.strip() for t in title_elem.itertext() if t and t.strip()).lower()

        sec_text = " ".join(t.strip() for t in sec.itertext() if t and t.strip())
        sections.append({
            "title": title,
            "text": sec_text.lower()
        })

    return sections

def extract_candidate_tables(article_xml_text):
    tables = []
    if not article_xml_text:
        return tables

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando XML para tablas] {e}")
        return tables

    for tbl in root.findall(".//{*}table-wrap"):
        label_elem = tbl.find(".//{*}label")
        caption_elem = tbl.find(".//{*}caption")

        label = " ".join(label_elem.itertext()).strip().lower() if label_elem is not None else ""
        caption = " ".join(caption_elem.itertext()).strip().lower() if caption_elem is not None else ""
        text = " ".join(t.strip() for t in tbl.itertext() if t and t.strip()).lower()

        tables.append({
            "label": label,
            "caption": caption,
            "text": text
        })

    return tables

def score_reference_as_included(ref, sections, tables):
    score = 0
    evidence = []

    first_author = ref.get("first_author")
    year = ref.get("year")

    if ref.get("pmid"):
        score += 2
        evidence.append("has_pmid")

    relevant_titles = [
        "included studies",
        "characteristics of included studies",
        "study characteristics",
        "included study",
        "results",
        "studies included",
        "study selection"
    ]

    for sec in sections:
        sec_title = sec["title"]
        sec_text = sec["text"]

        title_is_relevant = any(key in sec_title for key in relevant_titles)

        if first_author and year and first_author in sec_text and year in sec_text:
            if title_is_relevant:
                score += 5
                evidence.append(f"section:{sec_title}")
            else:
                score += 2
                evidence.append("author_year_in_section")

    table_keywords = [
        "included",
        "characteristics",
        "study",
        "studies"
    ]

    for tbl in tables:
        tbl_meta = f"{tbl['label']} {tbl['caption']}"
        tbl_text = tbl["text"]
        table_is_relevant = any(k in tbl_meta for k in table_keywords)

        if first_author and year and first_author in tbl_text and year in tbl_text:
            if table_is_relevant:
                score += 6
                evidence.append(f"table:{tbl_meta[:80]}")
            else:
                score += 3
                evidence.append("author_year_in_table")

    return score, evidence

def extract_included_study_candidates(article_xml_text, min_score=5):
    refs = extract_references_from_pmc_xml(article_xml_text)
    sections = extract_candidate_sections(article_xml_text)
    tables = extract_candidate_tables(article_xml_text)

    included = []
    all_scored = []

    for ref in refs:
        score, evidence = score_reference_as_included(ref, sections, tables)

        item = {
            **ref,
            "score": score,
            "evidence": evidence
        }
        all_scored.append(item)

        if score >= min_score:
            included.append(item)

    included = sorted(included, key=lambda x: x["score"], reverse=True)
    all_scored = sorted(all_scored, key=lambda x: x["score"], reverse=True)

    return included, all_scored

4.2) Test included-study detection + build mappings

In [ ]:
from collections import defaultdict
import time

test_pmcid = "PMC12835450"

oai_xml = fetch_pmc_xml_oai(test_pmcid)
print(oai_xml[:500] if oai_xml else "No OAI XML")

article_xml = extract_article_xml_from_oai(oai_xml)
print(article_xml[:500] if article_xml else "No article XML")

refs = extract_references_from_pmc_xml(article_xml)
print("N refs:", len(refs))
print("First refs:", refs[:3])

included, all_scored = extract_included_study_candidates(article_xml, min_score=5)
print("Included candidates:", len(included))
print("Top included candidates:", included[:5])

meta_included_candidates = defaultdict(list)
meta_all_scored_refs = defaultdict(list)

for d in meta_docs:
    pmid = d.get("pmid")
    pmcid = meta_to_pmc.get(pmid)

    if pmcid:
        oai_xml = fetch_pmc_xml_oai(pmcid)
        article_xml = extract_article_xml_from_oai(oai_xml)

        included, all_scored = extract_included_study_candidates(article_xml, min_score=5)

        meta_included_candidates[pmid] = included
        meta_all_scored_refs[pmid] = all_scored

        print(f"Meta PMID {pmid}: included candidates = {len(included)} / all refs = {len(all_scored)}")
        time.sleep(0.34)

print("Example meta PMID:", next(iter(meta_included_candidates)) if meta_included_candidates else "No meta")




<OAI-PMH xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/ http://www.openarchives.org/OAI/2.0/OAI-PMH.xsd">
    <responseDate>2026-04-21T15:33:37Z</responseDate>
    <request verb="GetRecord" identifier="oai:pubmedcentral.nih.gov:12835450" metadataPrefix="pmc">https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/</request>
    
    <GetRecord>
        <record>
    <header>
    <identifier
<ns0:article xmlns:ns0="https://jats.nlm.nih.gov/ns/archiving/1.4/" xmlns:ns2="http://www.niso.org/schemas/ali/1.0/" xmlns:ns3="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="https://jats.nlm.nih.gov/ns/archiving/1.4/ https://jats.nlm.nih.gov/archiving/1.4/xsd/JATS-archivearticle1-4.xsd" xml:lang="en" article-type="research-article" dtd-version="1.4"><ns0:processing-meta base-tagset="archiving" mathml-version="3.0" table-model="xhtml" tag

5) Collect cited PMIDs from references

In [ ]:
included_pmids = set()

for meta_pmid, refs in meta_included_candidates.items():
    for ref in refs:
        if ref.get("pmid"):
            included_pmids.add(str(ref["pmid"]))

included_pmids = sorted(included_pmids)

print("Included-study candidate PMIDs found:", len(included_pmids))
print(included_pmids[:10])

Included-study candidate PMIDs found: 47
['10183310', '14412840', '14706761', '16737531', '18395723', '19339127', '19631508', '21163595', '21372745', '22048261']


5.2) Save included PMIDs + prepare new PMIDs

In [ ]:
import json

# 1) Guardar todos los PMIDs incluidos (heurísticos)
with open("included_pmids.json", "w", encoding="utf-8") as f:
    json.dump(included_pmids, f, ensure_ascii=False, indent=2)

# 2) PMIDs semilla (metaanálisis)
seed_pmids = {
    str(d["pmid"])
    for d in meta_docs
    if d.get("pmid")
}

# 3) Filtrar: quitar los metaanálisis
new_included_pmids = [
    str(pmid) for pmid in included_pmids
    if str(pmid) not in seed_pmids
]

# 4) Ordenar
new_included_pmids = sorted(set(new_included_pmids))

# 5) Guardar
with open("new_included_pmids.json", "w", encoding="utf-8") as f:
    json.dump(new_included_pmids, f, ensure_ascii=False, indent=2)

print("Seed PMIDs:", len(seed_pmids))
print("All included PMIDs:", len(included_pmids))
print("New included PMIDs:", len(new_included_pmids))
print(new_included_pmids[:10])

Seed PMIDs: 10
All included PMIDs: 47
New included PMIDs: 47
['10183310', '14412840', '14706761', '16737531', '18395723', '19339127', '19631508', '21163595', '21372745', '22048261']


6) Fetch cited papers

In [ ]:
included_root = pubmed_fetch_xml(new_included_pmids[:100])
included_docs_raw = parse_pubmed_articles(included_root)

included_docs = []
for d in included_docs_raw:
    included_docs.append({
        "id": f"included_{d['pmid']}" if d.get("pmid") else d.get("id"),
        "pmid": d.get("pmid"),
        "type": "included-study-candidate",
        "title": d.get("title"),
        "text": d.get("text")
    })

print("Included-study candidate docs:", len(included_docs))
print(included_docs[0]["title"] if included_docs else "No docs")

Included-study candidate docs: 47
The MOS 36-item short form health survey. A conceptual analysis.


7) Final corpus + repository files

In [ ]:
import json

seen = set()
docs = []

for d in meta_docs + included_docs:
    pmid = d.get("pmid")
    if pmid and pmid not in seen:
        docs.append(d)
        seen.add(pmid)

print("TOTAL DOCS IN CORPUS:", len(docs))

# guardar corpus
with open("corpus_meta_plus_included.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

# guardar mapping meta → included candidates
with open("meta_included_map.json", "w", encoding="utf-8") as f:
    json.dump(dict(meta_included_candidates), f, ensure_ascii=False, indent=2)

print("Saved corpus_meta_plus_included.json")
print("Saved meta_included_map.json")

TOTAL DOCS IN CORPUS: 57
Saved corpus_meta_plus_included.json
Saved meta_included_map.json


8) Chunking

In [ ]:
import re

def chunk_text_sentences(text, chunk_chars=1200, overlap_chars=200):
    if not text or not text.strip():
        return []

    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    cur = ""

    for s in sents:
        s = s.strip()
        if not s:
            continue

        # si una sola frase ya supera el tamaño, la troceamos de forma defensiva
        if len(s) > chunk_chars:
            if cur:
                chunks.append(cur)
                cur = ""

            start = 0
            while start < len(s):
                end = start + chunk_chars
                piece = s[start:end]
                chunks.append(piece)
                start += max(1, chunk_chars - overlap_chars)
            continue

        if len(cur) + len(s) + 1 <= chunk_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = s

    if cur:
        chunks.append(cur)

    out = []
    for i, c in enumerate(chunks):
        if i == 0:
            out.append(c)
        else:
            prev = out[-1]
            overlap = prev[-overlap_chars:] if len(prev) > overlap_chars else prev
            out.append((overlap + " " + c).strip())

    return out


# reconstruir chunks con metadatos útiles para RAG
chunks = []

for d in docs:
    doc_text = d.get("text", "")
    doc_chunks = chunk_text_sentences(doc_text, chunk_chars=1200, overlap_chars=200)

    for i, c in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{d['id']}_chunk_{i}",
            "source_id": d.get("id"),
            "pmid": d.get("pmid"),
            "type": d.get("type"),
            "source_type_priority": 2 if d.get("type") == "meta-analysis" else 1,
            "title": d.get("title"),
            "chunk_index": i,
            "text": c
        })

print("N chunks:", len(chunks))
print("Chunk example:", chunks[0]["text"][:300] if chunks else "No chunks")

N chunks: 113
Chunk example: Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis. This study aimed to evaluate the effects of cognitive behavioral therapy for insomnia (CBT-I) on sleep quality and insomnia severity in menop


9) Embeddings + FAISS + retrieve()

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1) Modelo de embeddings
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2) Textos de los chunks
texts = [c["text"] for c in chunks]

# 3) Crear embeddings
emb = embed_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")

# 4) Crear índice FAISS con similitud coseno vía inner product
dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb)

print("Embeddings shape:", emb.shape)
print("FAISS index size:", index.ntotal)


# 5) Función de retrieval mejorada
def retrieve(query, k=5, initial_k=15, max_per_doc=1, use_type_priority=True):
    if not query or not query.strip():
        return []

    if len(chunks) == 0:
        return []

    initial_k = min(initial_k, len(chunks))
    k = min(k, len(chunks))

    q = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, idxs = index.search(q, initial_k)

    candidates = []
    for score, i in zip(scores[0], idxs[0]):
        ch = chunks[i]

        adjusted_score = float(score)
        if use_type_priority:
            adjusted_score += 0.02 * float(ch.get("source_type_priority", 1))

        candidates.append({
            "score": float(score),
            "adjusted_score": adjusted_score,
            "chunk_id": ch.get("chunk_id"),
            "source_id": ch.get("source_id"),
            "pmid": ch.get("pmid"),
            "type": ch.get("type"),
            "title": ch.get("title"),
            "chunk_index": ch.get("chunk_index"),
            "text": ch.get("text")
        })

    # ordenar por score ajustado
    candidates = sorted(candidates, key=lambda x: x["adjusted_score"], reverse=True)

    # limitar nº de chunks por documento
    results = []
    seen_docs = {}

    for cand in candidates:
        doc_id = cand["source_id"]
        count = seen_docs.get(doc_id, 0)

        if count >= max_per_doc:
            continue

        results.append(cand)
        seen_docs[doc_id] = count + 1

        if len(results) >= k:
            break

    return results


# 6) Test retrieval
res = retrieve(
    "What effects does cognitive behavioral therapy have on menopausal insomnia?",
    k=8,
    initial_k=20,
    max_per_doc=1,
    use_type_priority=True
)

for r in res:
    print(
        f"score={r['score']:.3f} | adj={r['adjusted_score']:.3f} | "
        f"type={r['type']} | pmid={r['pmid']} | source={r['source_id']}"
    )
    print("title:", r["title"])
    print(r["text"][:220], "\n---\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings shape: (113, 384)
FAISS index size: 113
score=0.757 | adj=0.797 | type=meta-analysis | pmid=41531400 | source=meta_41531400
title: Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis.
Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis. This study aimed to evaluate the effects of cognitive behavio 
---

score=0.766 | adj=0.786 | type=included-study-candidate | pmid=30481333 | source=included_30481333
title: Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene education.
Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene e

In [ ]:
# Pruebas pre-modelo

queries = [
    "How does CBT-I affect sleep quality in menopausal women?",
    "Does CBT-I improve insomnia severity during menopause?",
    "What evidence exists for CBT-I in postmenopausal women with vasomotor symptoms?"
]

for q in queries:
    print("\nQUERY:", q)
    res = retrieve(q, k=5, initial_k=15, max_per_doc=1, use_type_priority=True)
    for r in res:
        print(f"score={r['score']:.3f} | type={r['type']} | pmid={r['pmid']}")
        print("title:", r["title"])
        print("---")


QUERY: How does CBT-I affect sleep quality in menopausal women?
score=0.804 | type=included-study-candidate | pmid=33818845
title: The effect of group cognitive behavioural therapy for insomnia in postmenopausal women.
---
score=0.747 | type=included-study-candidate | pmid=30481333
title: Treating chronic insomnia in postmenopausal women: a randomized clinical trial comparing cognitive-behavioral therapy for insomnia, sleep restriction therapy, and sleep hygiene education.
---
score=0.721 | type=meta-analysis | pmid=41531400
title: Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis.
---
score=0.714 | type=included-study-candidate | pmid=31453958
title: Cognitive behavior therapy for menopausal symptoms (CBT-Meno): a randomized controlled trial.
---
score=0.694 | type=included-study-candidate | pmid=35191141
title: Effects of yoga on menopausal symptoms and sleep quality across me

10) Generative model: TinyLlama

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)

gen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32 if device == "cpu" else torch.float16,
    low_cpu_mem_usage=True
)

gen_model.to(device)
gen_model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Generative model loaded.")

Device: cpu


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Generative model loaded.


11) Full RAG (build_prompt + answer_rag)

In [ ]:
import torch

def build_context(results, max_chars=3200, max_per_doc=1):
    context_parts = []
    total_chars = 0
    seen_docs = {}

    for r in results:
        doc_id = r.get("source_id")

        count = seen_docs.get(doc_id, 0)
        if count >= max_per_doc:
            continue

        piece = (
            f"[TITLE] {r.get('title', 'Unknown title')}\n"
            f"[PMID] {r.get('pmid', 'NA')}\n"
            f"[TYPE] {r.get('type', 'unknown')}\n"
            f"[TEXT] {r.get('text', '')}\n"
        )

        if total_chars + len(piece) > max_chars:
            break

        context_parts.append(piece)
        total_chars += len(piece)
        seen_docs[doc_id] = count + 1

    return "\n\n".join(context_parts)


def build_prompt(question, contexts):
    context_text = build_context(contexts, max_chars=3200, max_per_doc=1)

    prompt = f"""You are a biomedical research assistant.

Answer the question using ONLY the provided context.
Do not invent facts.
Do not add general conclusions not present in the context.
Do not add generic concluding statements unless they are explicitly supported by the context.
If the context is insufficient, reply exactly: Not enough information.

Write a concise and factual answer in English.
When possible, summarize the evidence clearly and directly.

Question:
{question}

Context:
{context_text}

Answer:
"""
    return prompt


def generate_answer(prompt, max_new_tokens=180):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    if decoded.startswith(prompt):
        answer = decoded[len(prompt):].strip()
    else:
        answer = decoded.strip()

    return answer


def clean_answer(text):
    if not text:
        return ""

    text = text.replace("<|assistant|>", "").replace("<|user|>", "").replace("<|system|>", "").strip()

    lines = [line.strip() for line in text.split("\n") if line.strip()]
    cleaned_lines = []

    for line in lines:
        if line not in cleaned_lines:
            cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines).strip()

    stop_markers = ["Question:", "Context:", "Answer:"]
    for marker in stop_markers:
        if marker in cleaned:
            cleaned = cleaned.split(marker)[0].strip()

    generic_tails = [
        "However, further research is needed",
        "Further research is needed",
        "More research is needed",
        "Additional research is needed"
    ]
    for tail in generic_tails:
        if tail in cleaned:
            cleaned = cleaned.split(tail)[0].strip()

    return cleaned


def answer_rag(question, k=5, initial_k=15, max_per_doc=1, max_new_tokens=180):
    contexts = retrieve(
        question,
        k=k,
        initial_k=initial_k,
        max_per_doc=max_per_doc,
        use_type_priority=True
    )

    prompt = build_prompt(question, contexts)
    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    answer = clean_answer(answer)

    return {
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "prompt": prompt
    }


# Quick domain test

q = "Does CBT-I improve sleep quality and insomnia severity in menopausal women?"

rag_result = answer_rag(q, k=5, initial_k=15, max_per_doc=1, max_new_tokens=220)

print("QUESTION:\n", rag_result["question"])
print("\nANSWER:\n", rag_result["answer"])

print("\nTOP CONTEXTS:")
for c in rag_result["contexts"][:3]:
    print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")

Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
 Does CBT-I improve sleep quality and insomnia severity in menopausal women?

ANSWER:
 CBT-I is an effective non-pharmacological intervention for improving sleep quality and reducing insomnia severity in menopausal women. A meta-analysis of 52 weeks found that CBT-I significantly improved sleep quality (SMD=-1.01; 95% CI, -1.27 to -0.75) and reduced insomnia severity (MD=-4.49; 95% CI, -6.12 to -2.87). The effect size was moderate to large, indicating a significant improvement in sleep quality and insomnia severity. CBT-I was effective regardless of delivery mode (face-to-face or remote), follow-up duration, or baseline insomnia severity. The results suggest that CBT-I is an effective non-pharmacological intervention for menopausal women with insomnia. Standardized CBT-I training programs and clinical protocols should be developed to

TOP CONTEXTS:
- PMID=33818845 | type=included-study-candidate | title=The effect of group cognitive behavioural therapy for insomnia in postmen

12) Baseline vs RAG comparison

In [ ]:
# =========================
# 12) Baseline vs RAG comparison
# =========================

def answer_base(question, max_new_tokens=220):
    prompt = f"""You are a helpful biomedical assistant.
Answer the question clearly in 2-4 sentences.
If you are not sure, say so.

Question:
{question}

Answer:
"""
    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    answer = clean_answer(answer)
    return answer

queries = {
    "Incertidumbre diagnóstica": [
        "¿Cómo sé si lo que me está pasando es menopausia o no?",
        "¿Esto que me pasa es por la edad o por otra cosa?",
        "¿Todo lo que me pasa ahora tiene que ver con la menopausia?",
        "¿Cómo diferencio si un síntoma es hormonal o no?",
        "¿Puede ser solo estrés y no menopausia?",
        "¿Estoy en menopausia aunque no tenga todos los síntomas?",
        "¿Esto es normal en esta etapa o debería preocuparme?",
        "¿Hay alguna forma de saber con certeza qué cosas son de la menopausia y cuáles no?",
        "¿Todo lo que me pasa ahora lo estoy relacionando mal con la menopausia?",
        "¿Estoy exagerando o realmente me está pasando algo?"
    ],

    "Necesidad de orientación personalizada": [
        "¿Qué pruebas médicas debería hacerme ahora mismo?",
        "¿Qué tipo de especialista debería consultar?",
        "¿Qué seguimiento debería llevar en esta etapa?",
        "¿Qué analíticas son importantes en esta etapa de la menopausia?",
        "¿Cómo sé si me falta alguna vitamina?",
        "¿Qué debería revisar en mis análisis de sangre?",
        "¿Qué controles debería hacerme de forma preventiva?",
        "¿Cuándo debería hacerme una densitometría?",
        "¿Qué revisiones son realmente necesarias y cuáles no?",
        "¿Cómo puedo saber si estoy bien controlada médicamente?"
    ],

    "Demanda de acompañamiento": [
        "¿Por qué nadie me explica claramente qué me está pasando?",
        "¿Existe alguna forma de tener seguimiento diario de mis síntomas?",
        "¿Cómo sería tener un acompañamiento diario en esta etapa?",
        "¿Qué información necesito para tomar decisiones sobre mi salud?",
        "¿Cómo puedo tener una guía clara de qué hacer en cada momento?",
        "¿Por qué siento que voy a ciegas en este proceso?",
        "¿Cómo puedo organizar toda la información que recibo?",
        "¿Cómo puedo saber qué es importante y qué no?",
        "¿Cómo puedo tener una visión global de mi estado?"
    ],

    "Prevención": [
        "¿Qué debería haber hecho antes para prevenir esto?",
        "¿A qué edad debería haber empezado a cuidarme?",
        "¿Qué puedo hacer ahora para evitar empeorar?",
        "¿Es tarde para empezar a prevenir?",
        "¿Qué hábitos son clave en esta etapa?",
        "¿Cómo puedo prevenir problemas futuros?",
        "¿Qué debería cambiar en mi estilo de vida?",
        "¿Qué cosas empeoran los síntomas sin darme cuenta?",
        "¿Cómo influye la alimentación en esta etapa?",
        "¿Qué debería haber sabido antes?"
    ],

    "Cuerpo y cambios físicos": [
        "¿Por qué me están pasando cosas en el cuerpo que nunca me habían pasado?",
        "¿Es normal que cambie la piel de repente?",
        "¿Por qué tengo picores o cambios en la piel?",
        "¿Es normal tener más sequedad corporal?",
        "¿Por qué mi cuerpo reacciona diferente ahora?",
        "¿Esto que me pasa en el cuerpo es temporal?",
        "¿Mi cuerpo va a volver a ser como antes?",
        "¿Estos cambios son reversibles?",
        "¿Cuánto tiempo duran estos cambios físicos?",
        "¿Qué cambios son normales y cuáles no?",
        "¿De verdad estos cambios físicos dependen tanto de la genética?",
        "¿Me voy a parecer más a mi madre y a mi abuela materna que a la familia de mi padre en cómo llevo la menopausia?",
        "¿Por qué me engorda la barriga más que todo lo demás?",
        "¿Me puedo quedar medio calva o sin pelo por la menopausia?",
        "¿Por qué parece que las famosas no tienen menopausia o se la han “curado”?"
    ],

    "Síntomas específicos": [
        "¿Qué puedo hacer con los sofocos?",
        "¿Hay algo que funcione de verdad para el calor?",
        "¿Por qué tengo tanto calor de repente?",
        "¿Los sofocos se van o se quedan para siempre?",
        "¿Por qué algunas de mis amigas casi no tienen sofocos y yo sí?",
        "¿Qué puedo hacer con el insomnio?",
        "¿Por qué estoy más cansada que antes?",
        "¿Por qué me enfermo más que antes?",
        "¿Es normal tener más infecciones ahora?",
        "¿Por qué tengo dolores nuevos que antes no tenía?",
        "¿Por qué siento que mi sistema inmunológico está peor?",
        "¿Estos síntomas son para siempre o en algún momento aflojan?"
    ],

    "Medicación y sistema sanitario": [
        "¿Por qué me recetan tantas cosas sin explicarme bien para qué son?",
        "¿Estoy tomando demasiados medicamentos?",
        "¿La menopausia se está medicalizando demasiado?",
        "¿Qué alternativas tengo a tomar medicación para todo?",
        "¿Cuándo es realmente necesaria la terapia hormonal?",
        "¿Por qué a unas mujeres se les receta terapia hormonal y a otras no?",
        "¿La medicación para la menopausia puede aumentar el riesgo de cáncer?",
        "Si como bien, ¿de verdad tengo que tomar suplementos?",
        "¿Todas las mujeres deberían tomar suplementos o tratamiento hormonal, salvo contraindicaciones?",
        "¿Tiene límite de tiempo el tratamiento hormonal sustitutivo?",
        "¿Es mejor tratar síntomas sueltos o ir a la causa?",
        "¿Cómo puedo evitar tomar medicamentos innecesarios?",
        "¿Qué decisiones médicas dependen realmente de mí y no solo del médico?"
    ],

    "Límites de la IA": [
        "¿Puede una app decirme qué tomar?",
        "¿Hasta dónde puede ayudar una IA en mi salud?",
        "¿Cómo uso una app sin sustituir al médico de toda la vida?",
        "¿Qué tipo de recomendaciones son seguras en una app?",
        "¿Cómo combinar la información de una app con la atención médica?",
        "¿Qué puede hacer una IA y qué no?",
        "¿Cómo saber cuándo tengo que ir a un profesional de la salud?",
        "¿Puede una app orientarme sin diagnosticarme?",
        "¿Cómo evitar engancharme demasiado a una app o a la IA?",
        "¿Cómo usar la tecnología para cuidarme mejor sin hacer locuras?"
    ],

    "Emoción y estado de ánimo": [
        "¿Cómo afecta la menopausia a mi estado emocional?",
        "¿Por qué estoy más irritable?",
        "¿Por qué me afectan más las cosas que antes?",
        "¿Esto emocional es hormonal o psicológico?",
        "¿Cómo influye el estrés en la menopausia?",
        "¿Por qué siento que no soy la misma?",
        "¿Cómo puedo gestionar mejor mis emociones?",
        "¿Esto es pasajero o se queda?",
        "¿Cómo recuperar algo de estabilidad emocional?",
        "¿Qué relación hay entre menopausia y ansiedad?",
        "¿Por qué estoy tan triste si, en teoría, estaba deseando no volver a tener la regla?",
        "¿Los síntomas de la menopausia se parecen a los de mi madre o mi abuela? Me da miedo acabar igual que ellas."
    ],

    "Sexo, deseo y relaciones": [
        "¿Por qué no tengo ganas de sexo si mi pareja me sigue gustando?",
        "¿A todas las mujeres se les baja la libido en esta etapa?",
        "¿Hasta qué punto la bajada de deseo tiene que ver con el miedo al deterioro físico o a dejar de ser atractiva?",
        "¿La sequedad vaginal es para siempre o se puede mejorar?",
        "¿Hay cosas que ya no se pueden hacer en la cama en esta etapa?",
        "Si cambian las hormonas y sube la testosterona, ¿eso puede cambiar lo que me atrae o de quién me enamoro?"
    ],

    "Niebla mental, memoria y cansancio cognitivo": [
        "¿Es normal que se me olviden palabras que tengo 'en la punta de la lengua' todo el rato?",
        "¿Por qué tengo la sensación de que estoy más lenta para pensar que antes?",
        "¿Esto de la 'niebla mental' es cosa de la menopausia o es que me estoy volviendo torpe?",
        "¿Cómo sé si estos despistes son normales o si deberían preocuparme?",
        "¿Por qué me cuesta más concentrarme en cosas que antes hacía sin pensar?",
        "¿Es normal acabar el día con la cabeza agotada aunque no haya hecho 'tanto'?",
        "¿Voy a recuperar mi agilidad mental o esto se queda así?",
        "¿La menopausia puede afectar a mi memoria a largo plazo o solo a estos despistes del día a día?",
        "¿Qué puedo hacer para mejorar la concentración y no sentir que tengo la cabeza 'nublada'?",
        "¿Cómo explico a los demás que no es que sea despistada, sino que estoy notando cambios mentales reales?"
    ]
}


# =========================
# 12) Baseline vs RAG comparison
# =========================

for topic, topic_queries in queries.items():
    print(f"\nTopic: {topic}\n")
    for q in topic_queries:
        base = answer_base(q, max_new_tokens=220)
        rag_result = answer_rag(
            q,
            k=5,
            initial_k=15,
            max_per_doc=1,
            max_new_tokens=220
        )

        print("\n" + "=" * 80)
        print("QUESTION:", q)

        print("\nBASELINE ANSWER:")
        print(base)

        print("\nRAG ANSWER:")
        print(rag_result["answer"])

        print("\nTOP RAG SOURCES:")
        for c in rag_result["contexts"][:3]:
            print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")

Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Topic: Incertidumbre diagnóstica



Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: ¿Cómo sé si lo que me está pasando es menopausia o no?

BASELINE ANSWER:
Sí, yo sé.

RAG ANSWER:
Sure! Based on the provided context, we can conclude that the study investigated whether soy isoflavone supplementation could improve menopausal symptoms in perimenopausal and postmenopausal women. The results showed that the supplementation led to significant improvements in both the somatic and psychological domains, but not in the urogenital domain. The study also found that regular consumption of ViveSoy® was associated with a reduction in climacteric symptoms.

TOP RAG SOURCES:
- PMID=14412840 | type=included-study-candidate | title=Contemporary therapy of the menopausal syndrome.
- PMID=36276627 | type=included-study-candidate | title=The Impact of Soy Isoflavone Supplementation on the Menopausal Symptoms in Perimenopausal and Postmenopausal Women.
- PMID=26806546 | type=included-study-candidate | title=Impact of a soy drink on climacteric symptoms: an open-label, crossover

Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


QUESTION: ¿Mi cuerpo va a volver a ser como antes?

BASELINE ANSWER:
Sí, mi cuerpo va a volver a ser como antes.

RAG ANSWER:
The MOS 36-item short form health survey is a practical and widely-tested instrument for measuring health status and medical outcomes. The SF-36 is a conceptual analysis of this instrument, from the viewpoint of four models of health: medical, World Health Organization, wellness, and environmental. The authors suggest expanding the SF-36 to include additional dimensions of health, including incidents of morbidity, satisfaction with quality of life, absence of mental disorders, social adjustment, social support, higher states of functioning, and adjustment to the environment. The authors conclude that the SF-36 should be expanded to include these additional dimensions to better capture the full range of health and well-being.

TOP RAG SOURCES:
- PMID=10183310 | type=included-study-candidate | title=The MOS 36-item short form health survey. A conceptual analysis.

Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: ¿Estos cambios son reversibles?

BASELINE ANSWER:
Sí, los cambios son reversibles.

RAG ANSWER:
Sure! These changes are reversible. Based on the passage above, Can you provide a summary of the evidence supporting the effectiveness of a soy drink in reducing climacteric symptoms, including the impact on the Menopause Rating Scale and quality of life questionnaires?

TOP RAG SOURCES:
- PMID=26806546 | type=included-study-candidate | title=Impact of a soy drink on climacteric symptoms: an open-label, crossover, randomized clinical trial.
- PMID=8844630 | type=included-study-candidate | title=A menopause-specific quality of life questionnaire: development and psychometric properties.
- PMID=19339127 | type=included-study-candidate | title=Efficacy of soy isoflavones for depressive symptoms of the climacteric syndrome.


Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: ¿Cuánto tiempo duran estos cambios físicos?

BASELINE ANSWER:
Los cambios físicos pueden durar desde unos minutos hasta varios días.

RAG ANSWER:
The MOS 36-item short form health survey (SF-36) measures physical and mental health and well-being. It includes items such as pain, fatigue, and depression. The SF-36 has been used extensively in medical research, but not in psychological research. The authors propose expanding the SF-36 to include additional dimensions of health, including incidents of morbidity, satisfaction with quality of life, absence of mental disorders, social adjustment, social support, higher states of functioning, and adjustment to the environment. These proposed additions would increase the conceptual meaning of the SF-36 and make it more useful for medical research and surveys of the general population.

TOP RAG SOURCES:
- PMID=10183310 | type=included-study-candidate | title=The MOS 36-item short form health survey. A conceptual analysis.
- PMID=26806